# learn

> Fit the same linear head with fastai and write the file recognition reads.


In [ ]:
#| default_exp learn


In [ ]:
#| hide
from nbdev.showdoc import *


The Train button calls `nadakit.head.train_library`, which is the NumPy fit, and then ranks with `head.npz`. This notebook is the fastai version of that fit. It reads the embeddings the app already stored, trains `nn.Linear(768, n_sounds)` with cross-entropy and weight decay, and writes the same file. Install the extra environment first:

```sh
pixi install -e train
pixi run -e train jupyter lab nbs/14_learn.ipynb
```

The `train` environment does not include Modular MAX. The app keeps embedding on MAX either way.


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import numpy as np

from nadakit.head import save_head


In [ ]:
#| export
def train_fastai(X, y, ids, fp: str, path=None, epochs: int = 30, lr: float = 1e-2, wd: float = 0.1):
    """Fit a linear head with fastai and write `head.npz`.

    `X` is `(n, 768)`, `y` is `0 .. n_classes-1`, `ids` are the sound ids in that order,
    and `fp` is `fingerprint` of those sounds. Rows of the layer start at the class means.
    """
    import torch
    from fastai.data.core import DataLoaders
    from fastai.learner import Learner
    from fastai.losses import CrossEntropyLossFlat
    from torch import nn
    from torch.utils.data import TensorDataset

    X = np.asarray(X, np.float32)
    y = np.asarray(y, np.int64)
    xt = torch.tensor(np.ascontiguousarray(X))
    yt = torch.tensor(np.ascontiguousarray(y))
    ds = TensorDataset(xt, yt)
    dls = DataLoaders.from_dsets(ds, ds, bs=max(len(ds), 1))
    n = int(yt.max().item()) + 1
    model = nn.Linear(X.shape[1], n)
    with torch.no_grad():
        for k in range(n):
            take = xt[yt == k]
            if len(take):
                model.weight[k] = take.mean(0)
            model.bias[k] = 0
    learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), wd=wd)
    learn.fit_one_cycle(epochs, lr)
    save_head(model.weight.detach().cpu().numpy(), model.bias.detach().cpu().numpy(), ids, fp, path)
    return learn


Run the next cell in the `train` environment after the app has at least two taught sounds. It overwrites `data/head.npz`. The running app picks that file up on the next recognition, for as long as the library is unchanged.


In [ ]:
#| eval: false
from fastlite import database
from nadakit.core import data_dir
from nadakit.head import design, train_message
from nadakit.learn import train_fastai

class Sound:
    id: int = None
    name: str = ""
    glyph: str = ""
    key: str = ""
    samples: bytes = b""

rows = [s for s in database(str(data_dir() / "gnn.db")).create(Sound, pk="id")() if s.samples]
built = design(rows)
if built is None:
    print("Teach at least two different sounds before training.")
else:
    X, y, ids, fp = built
    train_fastai(X, y, ids, fp)
    print(train_message(int(y.size), len(ids)))


Unfreezing wav2vec is a different fit. `TorchEmbed` is the module, and it would need each example's waveform. Teaching stores the embedding and drops the audio, so a fine-tune cannot be replayed from the library, and a head fitted to today's vectors would not match an encoder that had moved. The Train button does not enter that path.
